# Load and structure in samples and labels

In [3]:
import numpy as np
import yaml
from pathlib import Path

# ---------------------------------------
# Basic config
# ---------------------------------------

root = Path("../../Datasets/Zhou2016/original/Zhou2016")

# Label mapping
# 0 = left hand
# 1 = right hand
# 2 = feet

class_mapping = {
    "left_hand": 0,
    "right_hand": 1,
    "feet": 2,
}

all_data = {}

# ---------------------------------------
# Loop over subjects and sessions
# ---------------------------------------

for subj in range(1, 5):   # 4 subjects

    subj_id = f"subject_{subj:02d}"
    all_data[subj_id] = {}

    for session in range(1, 4):   # 3 sessions

        session_id = f"session_{session:02d}"

        npz_path = root / f"{subj_id}_{session_id}.npz"
        yml_path = root / f"{subj_id}_{session_id}.yml"

        if not npz_path.exists() or not yml_path.exists():
            continue

        # Load EEG + stimulus
        npz = np.load(npz_path)

        eeg = npz["data"]      # (samples, channels), values in µV
        stim = npz["stim"]

        # Load metadata
        with open(yml_path, "r") as f:
            metadata = yaml.safe_load(f)

        fs = metadata["acquisition"]["samplingrate"]
        offset = metadata["stim"]["offset"]
        labels = metadata["stim"]["labels"]

        # Common classes
        code_mapping = {
            labels[name]: label
            for name, label in class_mapping.items()
        }

        # Non-zero stimulus positions
        event_samples = np.where(stim != 0)[0]

        X_all = []
        y_all = []

        for event_sample in event_samples:

            code = int(stim[event_sample])

            if code not in code_mapping:
                continue

            # Common 0.5–3.5 s MI analysis window
            start = event_sample + offset + int(0.5 * fs)
            end = event_sample + offset + int(3.5 * fs)

            trial = eeg[start:end, :].T

            if trial.shape[1] != int(3.0 * fs):
                continue

            # µV -> V
            trial = trial * 1e-6

            X_all.append(trial)
            y_all.append(code_mapping[code])

        X_all = np.stack(X_all)
        y_all = np.asarray(y_all)

        all_data[subj_id][session_id] = {
            "X": X_all,
            "y": y_all,
            "fs": fs,
        }

print("✅ Finished loading Zhou2016 3-class imagery dataset.")

✅ Finished loading Zhou2016 3-class imagery dataset.


# Filters and Feature extractions

In [4]:
for subj_id, sessions in all_data.items():

    for session_id, session_data in sessions.items():

        X = session_data["X"]
        y = session_data["y"]

        print(
            subj_id,
            session_id,
            "| X:", X.shape,
            "| y:", y.shape,
            "| classes:", np.unique(y, return_counts=True)
        )

subject_01 session_01 | X: (179, 14, 750) | y: (179,) | classes: (array([0, 1, 2]), array([60, 59, 60]))
subject_01 session_02 | X: (150, 14, 750) | y: (150,) | classes: (array([0, 1, 2]), array([50, 50, 50]))
subject_01 session_03 | X: (150, 14, 750) | y: (150,) | classes: (array([0, 1, 2]), array([50, 50, 50]))
subject_02 session_01 | X: (150, 14, 750) | y: (150,) | classes: (array([0, 1, 2]), array([50, 50, 50]))
subject_02 session_02 | X: (135, 14, 750) | y: (135,) | classes: (array([0, 1, 2]), array([45, 45, 45]))
subject_02 session_03 | X: (150, 14, 750) | y: (150,) | classes: (array([0, 1, 2]), array([50, 50, 50]))
subject_03 session_01 | X: (150, 14, 750) | y: (150,) | classes: (array([0, 1, 2]), array([50, 50, 50]))
subject_03 session_02 | X: (151, 14, 750) | y: (151,) | classes: (array([0, 1, 2]), array([50, 50, 51]))
subject_03 session_03 | X: (150, 14, 750) | y: (150,) | classes: (array([0, 1, 2]), array([50, 50, 50]))
subject_04 session_01 | X: (135, 14, 750) | y: (135,) |

In [5]:
from scipy.signal import butter, sosfiltfilt
from copy import deepcopy


def band_filter(data, fs, band=(4, 40), order=5):

    nyq = fs / 2.0

    low = band[0] / nyq
    high = band[1] / nyq

    sos = butter(
        order,
        [low, high],
        btype="band",
        output="sos"
    )

    return sosfiltfilt(
        sos,
        data,
        axis=-1
    )


filtered_data = deepcopy(all_data)


for subj_id, sessions in all_data.items():

    for session_id, session_data in sessions.items():

        X = session_data["X"]
        y = session_data["y"]
        fs = session_data["fs"]

        X_filt = band_filter(
            X,
            fs=fs,
            band=(4, 40),
            order=5
        )

        filtered_data[subj_id][session_id]["X"] = X_filt
        filtered_data[subj_id][session_id]["y"] = y


print("✅ All epochs filtered (4–40 Hz).")

✅ All epochs filtered (4–40 Hz).


In [6]:
import pandas as pd
from numpy.fft import fft
from tqdm import tqdm


def compute_time_cov(matrix):
    return np.cov(matrix)


def compute_freq_cov(matrix):

    fft_vals = np.abs(
        fft(matrix, axis=-1)
    )

    return np.cov(fft_vals)


def flatten_covariance(cov, prefix):

    idx = np.triu_indices_from(cov)

    vals = cov[idx]

    names = [
        f"{prefix}{i}_{j}"
        for i, j in zip(idx[0], idx[1])
    ]

    return vals, names


all_features = []


for subj_id, sessions in tqdm(
    filtered_data.items(),
    desc="Subjects"
):

    for session_id, session_data in sessions.items():

        X = session_data["X"]
        y = session_data["y"]

        for trial_idx, trial in enumerate(X):

            cov_t = compute_time_cov(trial)

            cov_t_vals, cov_t_names = flatten_covariance(
                cov_t,
                prefix="time_"
            )

            cov_f = compute_freq_cov(trial)

            cov_f_vals, cov_f_names = flatten_covariance(
                cov_f,
                prefix="freq_"
            )

            feature_vals = np.concatenate([
                cov_t_vals,
                cov_f_vals
            ])

            feature_names = (
                cov_t_names +
                cov_f_names
            )

            all_features.append({
                "subject": subj_id,
                "session": session_id,
                "label": int(y[trial_idx]),
                **{
                    feature_names[i]: feature_vals[i]
                    for i in range(len(feature_vals))
                }
            })


df_features = pd.DataFrame(all_features)

print("Shape:", df_features.shape)

df_features.head()

Subjects: 100%|██████████| 4/4 [00:00<00:00,  6.81it/s]


Shape: (1800, 213)


,subject,session,label,time_0_0,time_0_1,time_0_2,time_0_3,time_0_4,time_0_5,time_0_6,...,freq_10_10,freq_10_11,freq_10_12,freq_10_13,freq_11_11,freq_11_12,freq_11_13,freq_12_12,freq_12_13,freq_13_13
0,subject_01,session_01,2,1.137461e-09,1.063545e-09,3.196806e-10,2.775903e-10,3.080736e-10,2.243235e-10,2.034019e-10,...,3.062763e-08,2.602701e-08,2.624890e-08,2.545861e-08,3.289111e-08,2.952100e-08,2.755589e-08,3.094917e-08,2.938062e-08,3.008827e-08
1,subject_01,session_01,0,3.780096e-10,3.681056e-10,1.082588e-10,9.722319e-11,1.114552e-10,8.264429e-11,6.831956e-11,...,1.645922e-08,1.244656e-08,1.249538e-08,1.300504e-08,1.388293e-08,1.247688e-08,1.242206e-08,1.338329e-08,1.326365e-08,1.438244e-08
2,subject_01,session_01,0,1.357816e-10,1.219918e-10,6.060248e-11,5.493030e-11,4.820403e-11,4.872829e-11,4.665939e-11,...,2.719400e-08,1.697375e-08,1.444884e-08,1.491421e-08,1.899500e-08,1.548270e-08,1.494290e-08,1.522541e-08,1.481668e-08,1.590589e-08
3,subject_01,session_01,1,2.702881e-10,2.586604e-10,8.202128e-11,7.788849e-11,9.367278e-11,5.943425e-11,5.867707e-11,...,2.406223e-08,1.697434e-08,1.569604e-08,1.629081e-08,1.987422e-08,1.529449e-08,1.576156e-08,1.515265e-08,1.546920e-08,1.744643e-08
4,subject_01,session_01,2,2.123014e-11,1.904025e-11,1.862514e-11,1.897029e-11,1.704974e-11,1.763320e-11,1.741766e-11,...,1.464388e-08,1.362792e-08,1.288179e-08,1.316011e-08,1.799642e-08,1.572114e-08,1.601620e-08,1.596211e-08,1.613108e-08,1.761556e-08


In [7]:
df_features["label"].value_counts().sort_index()

label
0    600
1    599
2    601
Name: count, dtype: int64

In [8]:
df_features["subject"].value_counts().sort_index()

subject
subject_01    479
subject_02    435
subject_03    451
subject_04    435
Name: count, dtype: int64

In [9]:
df_features.groupby(
    ["subject", "session"]
).size()

subject     session   
subject_01  session_01    179
            session_02    150
            session_03    150
subject_02  session_01    150
            session_02    135
            session_03    150
subject_03  session_01    150
            session_02    151
            session_03    150
subject_04  session_01    135
            session_02    150
            session_03    150
dtype: int64

# Save

In [10]:
output_path = Path(
    "../../Datasets/Zhou2016/processed/Zhou2016_features.csv"
)

if not output_path.parent.exists():
    raise FileNotFoundError(
        f"Processed directory does not exist: {output_path.parent.resolve()}"
    )

df_features.to_csv(
    output_path,
    index=False
)

print(f"✅ Saved to: {output_path}")

✅ Saved to: ../../Datasets/Zhou2016/processed/Zhou2016_features.csv
